# Task 1

<p align="center">
  <img src="resources/img/aufgabe1_img.png" alt="Schwingungssystem" width="500">
</p>

The depicted oscillatory system, consisting of a homogeneous cylinder (radius $r$, mass $m$) and a spring-damper element (spring stiffness $c$, damping constant $d$), rolls without slipping on a plane. A harmonic force $F(t)=\hat{F}cos⁡\Omega t$ is applied at the center of the cylinder.

### Tasks
1. Isolate the body in its displaced position and indicate all forces and moments acting on it.

2. Derive the equation of motion for the system. Use the general form: $\ddot{q}+2\delta \dot{q} + \omega_{0}^{2}q = b_{0}u+b_{1}\dot{u} + b_{2} \ddot{u}$

3. Determine the amplitude response $V(\Omega)$ and the phase response $\Phi(\Omega)$ of the system. Sketch the amplitude response.

Given: $r,m,c,d,F(t)=\hat{F}cos\Omega⁡ tr$

In [106]:
%matplotlib widget
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [107]:
import math
import numpy as np
from scipy.integrate import odeint
from scipy import signal as signal

In [108]:
START_DEFLECTION = math.pi / 18  
START_VELOCITY = 0  
MASS = 1  
DEFAULT_C = 1  
DEFAULT_D = 0.1  
RADIUS = 1
OMEGA = 0.815 

In [109]:
class IntSolver:
    def __init__(self) -> None:
        return None 

    # Dauerlauf    
    def state_space_settled(self, z, t, d, m, c, omega):
        delta = d / (3 * m)
        omega_0 = np.sqrt(2 * c / (3 * m))
        b0 = 2 / (3 * m)
        # Zustandsvektor 
        # ...

        # Zustands-DGL
        # ... 
        [x, x_p] = z  # Zustandsvektor
        z_p = [
            x_p,
            -2 * delta * x_p - omega_0**2 * x + b0 * np.cos(omega * t),
        ]  # Zustands-DGL
        return z_p

    # Hochlauf
    def state_space_accelerated(self, z, t, d, m, c, omega):
        delta = d / (3 * m)
        omega_0 = np.sqrt(2 * c / (3 * m))
        alpha = omega / 100
        b0 = 2 / (3 * m)
        [x, x_p] = z
        z_p = [
            x_p,
            -2 * delta * x_p - omega_0**2 * x + b0 * np.cos(0.5 * alpha * t**2),
        ]
        return z_p

    def integrate(self, func, t, start_deflection, start_velocity, *args):
        z0 = (start_deflection, start_velocity)
        return odeint(func=func, y0=z0, t=t, args=args)[:, 0]
    
# make an instance of IntSolver
calculator = IntSolver()

In [110]:
t = np.linspace(0, 20, 500)

# calculate solution with our equation and starting conditions 
solution = calculator.integrate(calculator.state_space_settled,
                                t,
                                START_DEFLECTION,
                                START_VELOCITY,
                                DEFAULT_D,
                                MASS,
                                DEFAULT_C,
                                OMEGA)

# calculate correct solution with predefined calculate function 
from src.calculations.int_solver import IntSolverAufgabe1 as CorrectIntSolver
correct_calculator = CorrectIntSolver()
correct_solution = correct_calculator.integrate(calculator.state_space_settled,
                                t,
                                START_DEFLECTION,
                                START_VELOCITY,
                                DEFAULT_D,
                                MASS,
                                DEFAULT_C,
                                OMEGA)

# compare the two solutions and see if they are similar enough
from src.calculations.int_solver import validate_solution
correct = validate_solution(solution=solution, correct_solution=correct_solution)

Yor solution is correct!


In [ ]:
%matplotlib widget
%load_ext autoreload
%autoreload 2

from src.gui.gui_aufgabe_1 import GUI
from src.anim.animations.aufgabe_1 import Aufgabe1


try: 
    if correct: 
        animation_instance = Aufgabe1(calculator=calculator)
        app = GUI(default_c=DEFAULT_C,
                default_d=DEFAULT_D,
                animation_instance=animation_instance)
        display(app.app_layout)
except: 
    print("Run cells above or check your IntSolver solution")


# Probleme
# unterer Graph Bode Diagramm ist nicht sichtbar
# Sperrung der Kontrollelemente funtkioniert nach einigen Durchläufen nicht mehr (Performance Issues wahrscheinlich)
# generell schelchte Performance
# Achsenlimits für oberer Graph im Bode Diagramm sind unvorteilhaft
# Pfeil für Kraftindikator 
